##### Import the libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
import re
import spacy
import nltk
from nltk.corpus import stopwords
import plotly.express as px
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\balog\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\balog\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

##### Load the dataset

In [2]:
df_clean = pd.read_csv(r'C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Customer Feedback Analysis\Data\cleaned_reviews.csv')


##### Extract top positive and negative keywords

- Split Positive and Negative Reviews

In [3]:
positive_reviews = df_clean[df_clean["sentiment"] == "Positive"]["processed_review"]

negative_reviews = df_clean[df_clean["sentiment"] == "Negative"]["processed_review"]

- Extract top positive keywords

In [4]:
tfidf = TfidfVectorizer(max_features=20,ngram_range=(1,1))
positive_matrix = tfidf.fit_transform(positive_reviews)
positive_keywords = pd.DataFrame({"positive_keywords": tfidf.get_feature_names_out(), "TFIDF Score": positive_matrix.sum(axis=0).A1})
positive_keywords = positive_keywords.sort_values(by="TFIDF Score",ascending=False)

positive_keywords

,positive_keywords,TFIDF Score
1,amazon,1140.093478
8,good,700.098866
17,service,674.768272
0,always,562.771049
9,great,546.897164
13,order,507.132084
4,customer,496.116622
6,delivery,487.329490
18,time,448.858693
16,product,417.942948


- Extract top negative keywords

In [5]:
tfidf = TfidfVectorizer(max_features=20,ngram_range=(1,1))
negative_matrix = tfidf.fit_transform(negative_reviews)
negative_keywords = pd.DataFrame({"negative_keywords": tfidf.get_feature_names_out(), "TFIDF Score": negative_matrix.sum(axis=0).A1})
negative_keywords = negative_keywords.sort_values(by="TFIDF Score",ascending=False)

negative_keywords

,negative_keywords,TFIDF Score
1,amazon,3560.014489
11,not,2395.178238
3,customer,1984.206335
12,order,1918.807559
8,get,1886.441495
15,service,1869.932907
10,item,1654.007596
7,do,1562.892113
6,delivery,1557.677221
17,time,1397.703181


##### Generate Bigram Analysis (2-Word Phrases)

- Bigram Analysis for postive reviews

In [6]:
bigram = CountVectorizer(ngram_range=(2,2), max_features=20)
positive_x = bigram.fit_transform(positive_reviews)
positive_drivers = pd.DataFrame({"positive_drivers": bigram.get_feature_names_out(), "Count": positive_x.sum(axis=0).A1})
positive_drivers = positive_drivers.sort_values(by="Count", ascending=False)

positive_drivers

,positive_drivers,Count
5,customer service,975
6,do not,570
2,amazon prime,334
19,use amazon,322
4,can not,309
13,love amazon,262
15,next day,177
14,never problem,167
0,amazon always,149
7,fast delivery,149


- Bigram Analysis for negative reviews

In [7]:
bigram = CountVectorizer(ngram_range=(2,2), max_features=20)
negative_x = bigram.fit_transform(negative_reviews)
negative_drivers = pd.DataFrame({"negative_drivers": bigram.get_feature_names_out(), "Count": negative_x.sum(axis=0).A1})
negative_drivers = negative_drivers.sort_values(by="Count", ascending=False)

negative_drivers

,negative_drivers,Count
9,do not,7124
7,customer service,6244
3,can not,3172
2,be not,1228
0,amazon customer,1120
1,amazon prime,1058
19,will not,1043
11,gift card,1010
13,next day,978
14,not even,877


- Combine Both Tables

In [8]:
positive_drivers["sentiment"] = "positive"
negative_drivers["sentiment"] = "negative"

positive_drivers = positive_drivers.rename(columns={"positive_drivers": "drivers"})
negative_drivers = negative_drivers.rename(columns={"negative_drivers": "drivers"})

drivers = pd.concat([positive_drivers, negative_drivers], ignore_index=True)
drivers

,drivers,Count,sentiment
0,customer service,975,positive
1,do not,570,positive
2,amazon prime,334,positive
3,use amazon,322,positive
4,can not,309,positive
5,love amazon,262,positive
6,next day,177,positive
7,never problem,167,positive
8,amazon always,149,positive
9,fast delivery,149,positive


- Save the table

In [9]:
drivers.to_excel("sentiment_drivers.xlsx", index=False)